In [6]:
import numpy as np
import torch
from torch.distributions import Categorical
import torch.utils.data as data_utils
import matplotlib.pyplot as plt

# Import your classes
from AlphaBetaBot import AlphaBetaBot
from policy import Policy

import sys

from connect4 import Connect4Env

In [7]:
def print_progress(iteration, total, prefix='', suffix='', decimals=1, length=40, fill='█'):
    """
    Call in a loop to create a terminal progress bar similar to TensorFlow/Keras.
    """
    percent = ("{0:." + str(decimals) + "f}").format(100 * (iteration / float(total)))
    filled_length = int(length * iteration // total)
    bar = fill * filled_length + '-' * (length - filled_length)
    
    # \r returns the cursor to the start of the line
    sys.stdout.write(f'\r{prefix} |{bar}| {iteration}/{total} [{percent}%] {suffix}')
    sys.stdout.flush()
    
    # Print New Line on Complete
    if iteration == total:
        sys.stdout.write('\n')


def test_agent(env_class, policy = None, policy_2: None|Policy|AlphaBetaBot = None, games=100, visual=False): # (made with gemini)
    env = env_class()
    ai_wins = 0
    
    print(f"Running {games} games against other Bot...")
    
    for game in range(games):
        obs, _ = env.reset()
        # AI plays as Player 1 (Blue)
        done = False
        
        while not done:
            if env.current_player == 1: # AI Turn
                # AI Logic
                if policy is not None:
                    ai_input = torch.from_numpy(preprocess_board(obs, 1)).float().unsqueeze(0)
                    with torch.no_grad():
                        logits = policy.forward(ai_input)[0]
                        mask = torch.tensor([0.0 if env.board[0][c] == 0 else -1e9 for c in range(env.cols)])
                        action = torch.argmax(logits + mask).item()
                else:
                    print("please give policy to check")
                    return
            else: # Random Turn
                if policy_2 is not None and isinstance(policy_2, Policy):
                    ai_input = torch.from_numpy(preprocess_board(obs, -1)).float().unsqueeze(0)
                    with torch.no_grad():
                        logits = policy_2.forward(ai_input)[0]
                        mask = torch.tensor([0.0 if env.board[0][c] == 0 else -1e9 for c in range(env.cols)])
                        action = torch.argmax(logits + mask).item()
                elif policy_2 is not None and isinstance(policy_2, AlphaBetaBot):
                    action = policy_2.get_action(obs, -1)
                    
                else:
                    # Random Logic
                    legal = [c for c in range(env.cols) if env.board[0][c] == 0]
                    action = np.random.choice(legal)
                
            obs, _, done, _, info = env.step(action)
            if visual:
                env.render()
            
        if info['winner'] == 1:
            if visual:
                print("AI Wins!")
            ai_wins += 1
            
    print(f"Sigma AI Win Rate vs Bot: {ai_wins}/{games} ({ai_wins/games*100}%)")

def preprocess_board(board, current_player):
    """
    Converts a (6, 7) board with [-1, 0, 1] 
    into a (2, 6, 7) tensor with [0, 1].
    
    Channel 0: My Pieces
    Channel 1: Enemy Pieces
    """
    # If current_player is -1, we flip the board signs so the AI always sees itself as '1'
    relative_board = board * current_player
    
    # Masks
    my_pieces = (relative_board == 1).astype(np.float32)
    enemy_pieces = (relative_board == -1).astype(np.float32)
    
    # Stack them
    # Result shape: (2, 6, 7)
    return np.stack([my_pieces, enemy_pieces])

In [8]:
# Define constants
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
env = Connect4Env()

ppo_policy = Policy(env.action_space.n, input_channels=2, board_height=6, board_width=7, ent_coef=0.03, conv_layers_channels=[32, 64, 32], fc_layer_sizes=[256, 256, 256]).to(device)
print(device)

cuda


# Training loop

In [101]:
# Optional: Load previous weights if architecture matches
ppo_policy.load_from_file("connect4_cnn_iter_500000.pth")

total_observations = []
total_actions = []
total_rewards = []
total_old_probs = []
total_advantages = []
objectives = []
loss = []

total_games = 1e6

temperature = 0.5 # how likley is the Ai to choose the best move (0.00001 is 100% the best move)


for i in range(int(total_games)):
    obs, info = env.reset()
    terminated = False
    
    observations = []
    actions = []
    rewards = []
    old_probs = []
    steps = 0
    ''' 
        trajectory behaviour should match:
        state0, action0, reward1
        as in we want the action that matches the state it was taken as and
        the reward that was given after following that action in that state.
    '''
    while not terminated and steps < 6*8:
        steps += 1
        
        # Do NOT flatten here. Multiply by current player for perspective.
        current_obs = preprocess_board(obs, env.current_player)
        observations.append(current_obs)

        with torch.no_grad():
            # Prepare input: (1, 2, 6, 7) for CNN
            tensor_input = torch.from_numpy(current_obs).unsqueeze(0).float().to(device)
            
            unmasked_probs = ppo_policy(tensor_input)[0]

            # Masking
            legal_moves_mask = torch.tensor([0.0 if env.board[0][c] == 0 else -1e9 for c in range(env.cols)]).to(device)
            masked_probs = unmasked_probs + legal_moves_mask
            
            dist = Categorical(logits=masked_probs/temperature)
            action = dist.sample()

            actions.append(action.item())
            old_probs.append(dist.log_prob(action))

        obs, _, terminated, _, info = env.step(int(action))

    # Reward Calculation
    gamma = 0.99
    discounts = [gamma**i for i in range(len(observations))][::-1] 

    if info["winner"] == 1:
        rewards = [(1 * discounts[i]) if i % 2 == 0 else (-1 * discounts[i]) for i in range(len(observations))]
    elif info["winner"] == -1:
        rewards = [(-1 * discounts[i]) if i % 2 == 0 else (1 * discounts[i]) for i in range(len(observations))]
    else:
        rewards = [0] * len(observations)
    

    # Batch Preparation
    observations_np = np.array(observations, dtype="float32") # Shape (N, 2, 6, 7)
    # old_probs_np = np.array(old_probs, dtype="float32")

    observations_tensor = torch.from_numpy(observations_np).to(device)
    actions_tensor = torch.tensor(actions, dtype=torch.float32).to(device)
    old_probs_tensor = torch.tensor(old_probs).to(device)
    rewards_tensor = torch.tensor(rewards, dtype=torch.float32).to(device)
    
    # Advantage
    advantage = ppo_policy.advantage(observations_tensor, rewards_tensor)
    
    # Batches for training
    total_observations.append(observations_tensor)
    total_actions.append(actions_tensor)
    total_rewards.append(rewards_tensor)
    total_old_probs.append(old_probs_tensor)
    total_advantages.append(advantage)

    # Logging
    if i % 100 == 0:
        objectives.append(ppo_policy.objective(observations_tensor, actions_tensor, old_probs_tensor, rewards_tensor, advantage).detach().item())
        loss.append(ppo_policy.value_function.loss(observations_tensor, rewards_tensor).detach().item())


    if i % 10000 == 0:
        print("final board:")
        env.render()
        print(f"at epoch {i}:")
        print("obj:", objectives[-1] if objectives else 0)
        print("loss:", loss[-1] if loss else 0)
        # check against a random bot!

    if i % 500 == 0:
        print_progress(
            i%10000, 
            10000, 
            prefix='Training:', 
            suffix=f'- Obj: {objectives[-1]:.4f} - Loss: {loss[-1]:.4f}', 
            length=30
        )
    # Optimization
    if i % 1000 == 0 and i > 0:
        # Concatenate list of tensors to create a database
        dataset = data_utils.TensorDataset(
            torch.cat(total_observations), 
            torch.cat(total_actions), 
            torch.cat(total_old_probs), 
            torch.cat(total_rewards), 
            torch.cat(total_advantages)
        )
        loader = data_utils.DataLoader(dataset, batch_size=500, shuffle=True)
        
        for _ in range(5):
            for batch_obs, batch_action, batch_old_prob, batch_reward, batch_advantage in loader:
                ppo_policy.optimizer_step(batch_obs, batch_action, batch_old_prob, batch_reward, batch_advantage)
            
        total_observations = []
        total_actions = []
        total_rewards = []
        total_old_probs = []
        total_advantages = []
    
    # Saving (by gemini)
    if i > 0 and i % 100000 == 0:
        save_path = f"connect4_cnn_iter_{i}.pth"
        torch.save(ppo_policy.state_dict(), save_path)
        test_agent(Connect4Env, policy=ppo_policy, policy_2=AlphaBetaBot(depth=6), games=1) # Uncomment if file exists
        print(f"Model state saved to {save_path}")

    if i > 0 and i % 50000 == 0:
        plt.figure()
        plt.plot(objectives)
        plt.title(f"Objective at step {i}")
        plt.savefig(f"training_graph_{i}.png")
        plt.close()
        print(f"Graph saved to training_graph_{i}.png")

Loading from: connect4_cnn_iter_500000.pth to cuda
SUCCESS: Model loaded.
final board:
              
              
              
              
              
              

at epoch 0:
obj: 0.2653859555721283
loss: 0.6173519492149353
Training: |------------------------------| 0/10000 [0.0%] - Obj: 0.2654 - Loss: 0.6174

KeyboardInterrupt: 

In [ ]:
import connect4


obs, info = env.reset()
terminated = False

observations = []
actions = []
rewards = []
old_probs = []

while not terminated:
    # steps += 1
    
    # Do NOT flatten here. Multiply by current player for perspective.
    current_obs = preprocess_board(obs, env.current_player)
    observations.append(current_obs)

    with torch.no_grad():
        # Prepare input: (1, 2, 6, 7) for CNN
        tensor_input = torch.from_numpy(current_obs).unsqueeze(0).float().to(device)
        
        unmasked_probs = ppo_policy(tensor_input)[0]

        # Masking
        legal_moves_mask = torch.tensor([0.0 if env.board[0][c] == 0 else -1e9 for c in range(env.cols)]).to(device)
        masked_probs = unmasked_probs + legal_moves_mask
        
        dist = Categorical(logits=masked_probs)
        action = dist.sample()
        # print(action)

        actions.append(action.item())
        old_probs.append(dist.log_prob(action))

    obs, _, terminated, _, info = env.step(int(action))

connect4.render_board(obs)